<a href="https://colab.research.google.com/github/oleg61/DataScient/blob/Learn_AI/%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1_%D0%9E%D1%81%D0%BD%D0%BE%D0%B2%D1%8B_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D0%BD%D0%BD%D1%8B%D1%85_%D1%81%D0%B5%D1%82%D0%B5%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ: Основы нейронных сетей

**Автор:** Воропаев О.С.

**Цель:** научиться обучать простейшую нейросетевую модель, практически эквивалентную **линейной регрессии**, методом **градиентного спуска** в PyTorch.

### Задание
Реализовать обучение нейронной сети из **одного нейрона** для предсказания стоимости жилья:
- [California Housing Prices](https://www.kaggle.com/datasets/camnugent/california-housing-prices)  
  (в ноутбуке — `sklearn.datasets.fetch_california_housing`, без ручной загрузки с Kaggle)

Boston House Prices в современных версиях `sklearn` удалён; California Housing — рекомендуемый аналог.

### План
1. Загрузить и подготовить данные  
2. Разделить на train / test  
3. Создать модель (`nn.Sequential` с одним `Linear`)  
4. Обучить на train  
5. Проверить качество на test  


## 1. Импорты


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
torch.manual_seed(42)
np.random.seed(42)


## 2. Загрузка и подготовка данных

Датасет **California Housing**: признаки районов → медианная цена жилья в блоке (в $1000).

Один нейрон с линейной активацией решает ту же задачу, что линейная регрессия:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

Признаки стандартизируем (`StandardScaler`) — так SGD сходится стабильнее.


In [ ]:
housing = fetch_california_housing()
X = housing.data.astype(np.float32)
y = housing.target.astype(np.float32)

feature_names = list(housing.feature_names)
print("Признаки:", feature_names)
print("X:", X.shape, "| y:", y.shape)
print("y: min={:.2f}, max={:.2f}, mean={:.2f}".format(y.min(), y.max(), y.mean()))


## 3. Train / Test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_t = torch.from_numpy(X_train).to(device)
y_train_t = torch.from_numpy(y_train).view(-1, 1).to(device)
X_test_t = torch.from_numpy(X_test).to(device)
y_test_t = torch.from_numpy(y_test).view(-1, 1).to(device)

print(f"train: {X_train_t.shape}, test: {X_test_t.shape}")

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=64,
    shuffle=True,
)


## 4. Модель: один нейрон

`nn.Linear(n_features, 1)` — ровно один нейрон без нелинейности.  
Через `nn.Sequential` — как просят в инструкции.


In [ ]:
n_features = X_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(n_features, 1)  # один нейрон = линейная регрессия
).to(device)

print(model)
print("Параметры:")
for name, p in model.named_parameters():
    print(f"  {name}: {tuple(p.shape)}")


## 5. Обучение (градиентный спуск)

- **Loss:** MSE (`nn.MSELoss`)
- **Оптимизатор:** SGD
- Цикл: forward → loss → `backward()` → `optimizer.step()`


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

n_epochs = 40
history = {"train_loss": [], "test_mse": []}

for epoch in range(1, n_epochs + 1):
    model.train()
    running = 0.0
    n_batches = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
        n_batches += 1

    train_loss = running / max(n_batches, 1)

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)
        test_mse = criterion(test_pred, y_test_t).item()

    history["train_loss"].append(train_loss)
    history["test_mse"].append(test_mse)

    if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
        print(
            f"Epoch {epoch:02d}/{n_epochs} | "
            f"train MSE: {train_loss:.4f} | test MSE: {test_mse:.4f}"
        )


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train MSE")
plt.plot(history["test_mse"], label="test MSE")
plt.xlabel("эпоха")
plt.ylabel("MSE")
plt.title("Обучение одного нейрона (California Housing)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Качество на тестовых данных


In [ ]:
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t).cpu().numpy().ravel()

y_true = y_test

mse = mean_squared_error(y_true, y_pred_t)
rmse = float(np.sqrt(mse))
mae = mean_absolute_error(y_true, y_pred_t)
r2 = r2_score(y_true, y_pred_t)

print("=== Test metrics (PyTorch, 1 нейрон) ===")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}   (в единицах y — $1000)")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")


In [ ]:
# Сравнение с Ordinary Least Squares (sklearn)
ols = LinearRegression().fit(X_train, y_train)
ols_pred = ols.predict(X_test)

print("R² PyTorch (1 нейрон + SGD):", round(r2, 4))
print("R² sklearn LinearRegression:", round(r2_score(y_true, ols_pred), 4))


In [ ]:
# Сравнение параметров: веса нейрона vs коэффициенты OLS
neuron_w = model[0].weight.detach().cpu().numpy().ravel()
neuron_b = float(model[0].bias.detach().cpu().numpy().ravel()[0])
ols_w = ols.coef_.ravel()
ols_b = float(ols.intercept_)

print(f"смещение (bias): нейрон = {neuron_b:.4f} | LinearRegression = {ols_b:.4f}")
print(f"максимальное расхождение весов: {np.max(np.abs(neuron_w - ols_w)):.2e}")
print()
print(f"{'признак':12s} {'вес нейрона (GD)':>18s} {'коэфф. LinearRegression':>24s}")
for name, w_n, w_o in zip(feature_names, neuron_w, ols_w):
    print(f"{name:12s} {w_n:18.4f} {w_o:24.4f}")


In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_true, y_pred_t, alpha=0.25, s=12)
lims = [min(y_true.min(), y_pred_t.min()), max(y_true.max(), y_pred_t.max())]
plt.plot(lims, lims, "r--", label="идеал y = ŷ")
plt.xlabel("y true ($1000)")
plt.ylabel("y pred ($1000)")
plt.title("Test: факт vs прогноз")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Выводы

По результатам прогона один нейрон (SGD) и `sklearn.LinearRegression` дали **практически одинаковые** параметры:

| Признак | вес нейрона (GD) | коэфф. LinearRegression |
|---------|-----------------:|------------------------:|
| MedInc | 0.7390 | 0.7390 |
| HouseAge | 0.1060 | 0.1060 |
| AveRooms | −0.2547 | −0.2546 |
| AveBedrms | 0.2934 | 0.2934 |
| Population | −0.0020 | −0.0020 |
| AveOccup | −0.0353 | −0.0353 |
| Latitude | −0.7757 | −0.7758 |
| Longitude | −0.7523 | −0.7523 |

- смещение (bias): нейрон ≈ **0**, LinearRegression ≈ **0** (после стандартизации признаков);
- максимальное расхождение весов: **3.04×10⁻⁵** — на уровне численной погрешности.

### Что это значит
1. Загрузили California Housing, стандартизировали признаки, разбили данные 80/20 на train/test.
2. Модель `nn.Sequential(nn.Linear(n_features, 1))` без нелинейности — это линейная регрессия: $\hat{y} = w^\top x + b$.
3. Обучили **SGD + MSE** на train; на test сравнили метрики и **сами веса** с OLS.
4. Цель задания выполнена: нейросеть из одного нейрона, обученная градиентным спуском в PyTorch, **эквивалентна** линейной регрессии — совпали не только прогнозы, но и коэффициенты.

**Под капотом:** `loss.backward()` считает $\partial L/\partial w$, $\partial L/\partial b$, а `optimizer.step()` обновляет веса: $w \leftarrow w - \mathrm{lr}\cdot\nabla_w L$.
